# MMScan data check

Exploratory pass over the freshly-downloaded MMScan / EmbodiedScan-v1 / EmbodiedScan-v2-beta data before writing any real preprocessing.
matterport3d raw scans are **not downloaded yet** -- cells below flag anything that touches it instead of failing.

In [1]:
import json, pickle, os
from pathlib import Path
import numpy as np

BASE = Path("/glob/g01-cache/pf/Yushuo/vjepa201")
MMSCAN_ROOT = BASE / "source_data/mmscan_data"
SPLIT_DIR = MMSCAN_ROOT / "embodiedscan_split"
V1_DIR = SPLIT_DIR / "embodiedscan-v1"
V2_DIR = SPLIT_DIR / "embodiedscan-v2"
BETA_DIR = MMSCAN_ROOT / "MMScan-beta"

RAW_ROOTS = {
    "scannet": BASE / "source_data/scannet",
    "3rscan": BASE / "source_data/3rscan",
    "matterport3d": BASE / "source_data/matterport3d",  # not downloaded yet
}

for name, p in {"mmscan_root": MMSCAN_ROOT, "v1": V1_DIR, "v2": V2_DIR, "beta": BETA_DIR, **RAW_ROOTS}.items():
    print(f"{'OK ' if p.exists() else 'MISSING '} {name:15s} {p}")

OK  mmscan_root     /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data
OK  v1              /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v1
OK  v2              /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v2
OK  beta            /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/MMScan-beta
OK  scannet         /glob/g01-cache/pf/Yushuo/vjepa201/source_data/scannet
OK  3rscan          /glob/g01-cache/pf/Yushuo/vjepa201/source_data/3rscan
MISSING  matterport3d    /glob/g01-cache/pf/Yushuo/vjepa201/source_data/matterport3d


## Directory tree (depth-limited)
Just to see what actually landed on disk before assuming any filenames.

In [2]:
def print_tree(root: Path, max_depth=3, max_items=15):
    if not root.exists():
        print(f"[missing] {root}")
        return
    root_depth = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        dirpath = Path(dirpath)
        depth = len(dirpath.parts) - root_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{dirpath.name}/")
        dirnames.sort()
        for d in dirnames[:max_items]:
            pass  # walked next iteration, just counting here
        if len(dirnames) > max_items:
            print(f"{indent}  ... ({len(dirnames)} subdirs total, showing first {max_items})")
            dirnames[:] = dirnames[:max_items]
        for f in sorted(filenames)[:max_items]:
            size = (dirpath / f).stat().st_size
            print(f"{indent}  {f}  ({size/1e6:.2f} MB)")
        if len(filenames) > max_items:
            print(f"{indent}  ... ({len(filenames)} files total, showing first {max_items})")

print_tree(MMSCAN_ROOT, max_depth=3, max_items=15)

mmscan_data/
  embodiedscan-v2-beta.zip  (367.93 MB)
  embodiedscan.zip  (304.47 MB)
  MMScan-beta/
    README.md  (0.01 MB)
    Data_splits/
      test-split.json  (0.03 MB)
      train-split.json  (0.12 MB)
      val-split.json  (0.03 MB)
    MMScan_meta/
      MMScan_object_meta.json  (92.03 MB)
      MMScan_region_meta.json  (28.59 MB)
    MMScan_samples/
      MMScan_Caption_object.json  (55.83 MB)
      MMScan_Caption_region.json  (7.84 MB)
      MMScan_QA.json  (1991.09 MB)
      MMScan_VG.json  (437.83 MB)
  embodiedscan_split/
    embodiedscan-v1/
      embodiedscan_infos_test.pkl  (57.50 MB)
      embodiedscan_infos_train.pkl  (320.88 MB)
      embodiedscan_infos_val.pkl  (85.05 MB)
      embodiedscan_test_vg.json  (11.18 MB)
      embodiedscan_train_mini_vg.json  (12.17 MB)
      embodiedscan_train_vg.json  (59.19 MB)
      embodiedscan_train_vg_all.json  (63.75 MB)
      embodiedscan_val_mini_vg.json  (3.00 MB)
      embodiedscan_val_vg.json  (14.47 MB)
      embodiedscan_v

## Generic file inspector
Loads json / pkl / npy files found under a root (capped) and prints type + top-level keys/shape, without assuming exact filenames -- those haven't all been confirmed yet.

In [3]:
def describe(obj, max_items=10):
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"  dict, {len(keys)} keys: {keys[:max_items]}")
        first_key = keys[0] if keys else None
        if first_key is not None:
            print(f"  obj[{first_key!r}] = {str(obj[first_key])[:300]}")
    elif isinstance(obj, list):
        print(f"  list, {len(obj)} items")
        if obj:
            print(f"  obj[0] = {str(obj[0])[:300]}")
    elif isinstance(obj, np.ndarray):
        print(f"  ndarray shape={obj.shape} dtype={obj.dtype}")
    else:
        print(f"  {type(obj)}: {str(obj)[:300]}")

def inspect_root(root: Path, max_files=8, size_limit_mb=200):
    if not root.exists():
        print(f"[missing] {root}")
        return
    seen = 0
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if seen >= max_files:
                return
            path = Path(dirpath) / f
            size_mb = path.stat().st_size / 1e6
            if size_mb > size_limit_mb:
                print(f"\n[skip, {size_mb:.0f}MB > limit] {path}")
                continue
            try:
                if f.endswith(".json"):
                    print(f"\n[json] {path}")
                    describe(json.load(open(path)))
                elif f.endswith(".pkl"):
                    print(f"\n[pkl] {path}")
                    describe(pickle.load(open(path, "rb")))
                elif f.endswith(".npy"):
                    print(f"\n[npy] {path}")
                    describe(np.load(path, allow_pickle=True))
                else:
                    continue
            except Exception as e:
                print(f"\n[error] {path}: {e}")
            seen += 1

In [4]:
print("=== embodiedscan-v1 ===")
inspect_root(V1_DIR, max_files=8)

=== embodiedscan-v1 ===

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v1/embodiedscan_train_vg_all.json
  list, 247468 items
  obj[0] = {'scan_id': 'scannet/scene0191_00', 'target_id': 5, 'distractor_ids': [6], 'text': 'the board that is beside the door', 'target': 'board', 'anchors': ['door'], 'anchor_ids': [3], 'tokens_positive': [[4, 9]]}

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v1/embodiedscan_val_mini_vg.json
  list, 11818 items
  obj[0] = {'scan_id': 'scannet/scene0072_01', 'target_id': 20, 'distractor_ids': [8, 28, 29], 'text': 'find the pillow that is on top of the shelf', 'target': 'pillow', 'anchors': ['shelf'], 'anchor_ids': [22], 'tokens_positive': [[9, 15]]}

[skip, 321MB > limit] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v1/embodiedscan_infos_train.pkl

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_d

In [5]:
print("=== embodiedscan-v2 ===")
inspect_root(V2_DIR, max_files=8)

=== embodiedscan-v2 ===

[skip, 560MB > limit] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v2/embodiedscan_infos_train.pkl

[pkl] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v2/embodiedscan_infos_test.pkl
  dict, 2 keys: ['metainfo', 'data_list']
  obj['metainfo'] = {'DATASET': 'SCANNET+3RSCAN+MP3D+ARKIT', 'categories': {'adhesive tape': 1, 'air conditioner': 2, 'alarm': 3, 'album': 4, 'arch': 5, 'backpack': 6, 'bag': 7, 'balcony': 8, 'ball': 9, 'banister': 10, 'bar': 11, 'barricade': 12, 'baseboard': 13, 'basin': 14, 'basket': 15, 'bathtub': 16, 'beam': 17, 'b

[pkl] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v2/embodiedscan_infos_val.pkl
  dict, 2 keys: ['metainfo', 'data_list']
  obj['metainfo'] = {'DATASET': 'SCANNET+3RSCAN+MP3D+ARKIT', 'categories': {'adhesive tape': 1, 'air conditioner': 2, 'alarm': 3, 'album': 4, 'arch': 5, 'backpack': 6, 

In [6]:
print("=== MMScan-beta-release ===")
inspect_root(BETA_DIR, max_files=8)

=== MMScan-beta-release ===

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/MMScan-beta/Data_splits/val-split.json
  list, 814 items
  obj[0] = matterport3d/ULsKaCPVFJR/region0

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/MMScan-beta/Data_splits/test-split.json
  list, 702 items
  obj[0] = matterport3d/EU6Fwq7SyZv/region0

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/MMScan-beta/Data_splits/train-split.json
  list, 3092 items
  obj[0] = matterport3d/ZMojNkEp431/region0

[json] /glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/MMScan-beta/MMScan_samples/MMScan_Caption_region.json
  dict, 2 keys: ['train', 'val']
  obj['train'] = [{'scan_id': 'matterport3d/ZMojNkEp431/region0', 'object_list': [13, 16, 17, 19, 20, 21, 22, 25, 27, 29, 37, 43, 45, 47, 48, 49, 50, 53, 56], 'region_id': '0', 'region_type': 'storage region', 'region_caption': 'The storage region is a multifunctional space designed for transition, storage, s

## Known asset: embodiedscan_occupancy (v1)
Already confirmed on disk: `embodiedscan_occupancy/<dataset>/<scene_id>/{occupancy.npy, visible_occupancy.pkl}`.

In [7]:
occ_root = V1_DIR / "embodiedscan_occupancy"
if occ_root.exists():
    for dataset_dir in sorted(occ_root.iterdir()):
        if not dataset_dir.is_dir():
            continue
        scene_dirs = list(dataset_dir.iterdir())
        print(f"{dataset_dir.name}: {len(scene_dirs)} scenes")
        if scene_dirs:
            sample_scene = scene_dirs[0]
            occ = np.load(sample_scene / "occupancy.npy", allow_pickle=True)
            vis = pickle.load(open(sample_scene / "visible_occupancy.pkl", "rb"))
            print(f"  sample: {sample_scene.name}")
            print(f"  occupancy.npy shape={occ.shape} dtype={occ.dtype}")
            describe(vis)
else:
    print(f"[missing] {occ_root}")

3rscan: 1178 scenes
  sample: bcb0fe19-4f39-2c70-9cb7-a6993671cc91
  occupancy.npy shape=(1368, 4) dtype=int64
  list, 107 items
  obj[0] = {'img_path': '3rscan/bcb0fe19-4f39-2c70-9cb7-a6993671cc91/sequence/frame-000000.color.jpg', 'visible_occupancy': array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
   
matterport3d: 61 scenes


FileNotFoundError: [Errno 2] No such file or directory: '/glob/g01-cache/pf/Yushuo/vjepa201/source_data/mmscan_data/embodiedscan_split/embodiedscan-v1/embodiedscan_occupancy/matterport3d/PX4nDJXEHrG/occupancy.npy'

## Scene-ID cross-check: annotation vs raw scans on disk
For each source dataset referenced by the occupancy folders, check how many of those scene IDs actually exist under the raw scan roots downloaded so far.

In [8]:
def raw_scene_ids(dataset_name):
    root = RAW_ROOTS.get(dataset_name)
    if root is None or not root.exists():
        return None
    scans_dir = root / "scans" if (root / "scans").exists() else root
    return {p.name for p in scans_dir.iterdir() if p.is_dir()}

if occ_root.exists():
    for dataset_dir in sorted(occ_root.iterdir()):
        if not dataset_dir.is_dir():
            continue
        ann_ids = {p.name for p in dataset_dir.iterdir() if p.is_dir()}
        raw_ids = raw_scene_ids(dataset_dir.name)
        if raw_ids is None:
            print(f"{dataset_dir.name}: {len(ann_ids)} annotated scenes, raw data NOT downloaded -- blocked")
            continue
        overlap = ann_ids & raw_ids
        print(f"{dataset_dir.name}: {len(ann_ids)} annotated, {len(raw_ids)} raw on disk, {len(overlap)} overlap "
              f"({len(ann_ids - raw_ids)} annotated scenes missing raw data)")

3rscan: 1178 annotated, 1381 raw on disk, 1178 overlap (0 annotated scenes missing raw data)
matterport3d: 61 annotated scenes, raw data NOT downloaded -- blocked
scannet: 1201 annotated, 1513 raw on disk, 1201 overlap (0 annotated scenes missing raw data)


## Optional: MMScan devkit (only if cloned + installed)
The devkit (`rbler1234/MMScan` repo, or the `mmscan` branch of `InternRobotics/EmbodiedScan`) is a separate install step -- this cell is a no-op until that's done.

In [ ]:
try:
    from mmscan import MMScan
    ds = MMScan(split="val", task="MMScan-VG")
    print(f"MMScan devkit loaded, {len(ds)} val/VG samples")
    print(ds[0])
except ImportError:
    print("mmscan devkit not installed yet -- skip (see data_preparation/README.md in the mmscan branch)")